# AAI-530 Assignment 1.1
## LLM Access Comparison Brief
## Overview

Large language models can be accessed in many ways, including commercial APIs, hosted open-weight models, and locally run models. Each access method involves tradeoffs related to cost, latency, ease of use, model quality, privacy, reproducibility, and deployment complexity.

In this assignment, you will use an AI assistant of your choice to help you build a small LLM-powered application that compares two different model access methods. Rather than completing a highly scaffolded coding worksheet, you will work from a set of required outcomes You should use AI tools to help write, debug, and revise your code, but you are responsible for understanding, testing, documenting, and explaining the final result.

Because AI assistance is expected, your grade will focus less on whether you personally wrote every line of code and more on whether your final workflow runs, produces meaningful outputs, supports a clear comparison, and is explained thoughtfully in your written responses. Make sure that you complete all work listed under the **TODO:** for each assignment section, and answer all written questions in the provided cells marked **A:** with complete, thoughtful responses that directly refer to your code output and experiences.

By the end of this assignment, you should be able to:

1. Connect to at least one LLM through an API or hosted inference endpoint.
2. Send prompts to two different LLM access methods or model configurations.
3. Compare the outputs from both systems on the same task.
4. Evaluate model access options in terms of quality, cost, latency, usability, and suitability for an applied task.

In [ ]:
# Use this cell to import libraries, set up utility functions, define save/load locations, etc.

# This notebook runs locally so that Model 2 can be a local Ollama model.
%pip install -q google-genai ollama pandas


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import libraries used throughout the notebook.
# Credentials are read from environment variables (or prompted for) rather than hardcoded.

import os
import time
import getpass
import pandas as pd

from google import genai
import ollama

# Model 1 (Gemini) needs an API key. Model 2 (Ollama) runs locally and needs no key.
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or getpass.getpass("Enter GEMINI_API_KEY: ")
assert GEMINI_API_KEY, "GEMINI_API_KEY is missing."

# Check that the local Ollama server is reachable and list the models that are installed.
try:
    local_models = [m.model for m in ollama.list().models]
    print("Ollama server reachable. Installed models:", local_models)
except Exception as e:
    raise RuntimeError(
        "Could not reach Ollama at http://localhost:11434. "
        "Run `ollama serve` (or open the Ollama app) and try again."
    ) from e

print("Gemini credential loaded; Ollama runtime available.")


Ollama server reachable. Installed models: ['llama3.1:8b']
Gemini credential loaded; Ollama runtime available.


## Part 1: Model Access Setup

**TODO:**
- Connect to your first model access method.
- Then connect to your second model access method, model, or configuration.

### Suggested Access Options

You may compare any two reasonable options, such as:
- Gemini API and Hugging Face Inference API
- OpenAI API and Hugging Face Inference API
- Claude API and a locally hosted model
- Two different hosted models through the same provider
- A hosted API model and a local model run through Ollama or transformers
- Two different configurations of the same model, such as different temperature settings, if access to multiple models is not feasible

If you are unable to access a paid API, you may use a free or locally available model. If you run into access problems that prevent you from completing the comparison as planned, document the issue clearly and complete the assignment using an alternative model or configuration.

**Q1:** For each access method, briefly document:
- provider or tool used
- model name
- whether the model is accessed through an API, hosted endpoint, local runtime, or user-facing assistant
- any setup challenges you encountered

**A1:**

**Model 1 — Google Gemini (`gemini-2.5-flash`).** Accessed through the Gemini API using Google's `google genai` SDK and an API key. The model runs on Google's servers; every prompt is sent over the network.

**Model 2 — Ollama (`llama3.1:8b`, Meta's Llama 3.1 8B Instruct, 4-bit quantized).** Accessed through a *local runtime*: the Ollama server runs on my Mac and exposes an HTTP API on `localhost:11434`, which the `ollama` Python package calls. No API key, no credits, and no data leaves the machine.

**Setup challenges.** My original plan was to use the Hugging Face Inference API as Model 2, but the request to `meta-llama/Llama-3.1-8B-Instruct` returned a 403 (the model is gated and I had no inference credits). I switched to a local model instead, which the assignment permits. For Ollama the setup was installing the app, running `ollama pull llama3.1:8b` (a ~5 GB download), and making sure the server was running before executing the notebook. Because the notebook now depends on a local process, it can no longer run in Google Colab, so I replaced `google.colab.userdata` with environment variables.


**Q2:** What prompt or prompts did you give your AI assistant to help set up the model connections? What did you need to modify or verify yourself?

**A2:**

I used AI to help identify the required libraries, configure the API clients, and organize the notebook into reusable code cells. I still needed to install the dependencies, enter my own API credentials, verify the model identifiers, and test the requests. I also needed to review any errors and confirm that the returned responses came from the intended models.

In [ ]:
# Code for PART 1

# This is a cloud-vs-local comparison:
#   Model 1 is a hosted commercial API (Gemini).
#   Model 2 is an open-weight model running on this machine via Ollama.

GEMINI_MODEL = "gemini-2.5-flash"
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

# Pick a local model that is already pulled. llama3.1:8b needs ~16 GB RAM;
OLLAMA_MODEL = "llama3.1:8b"

assert any(m.startswith(OLLAMA_MODEL) for m in local_models), (
    f"{OLLAMA_MODEL} is not installed. Run: ollama pull {OLLAMA_MODEL}"
)

# Warm up the local model once so the first measured latency does not include loading weights into memory.
_ = ollama.chat(model=OLLAMA_MODEL, messages=[{"role": "user", "content": "hi"}])

print("Model clients configured.")
print("Gemini model:", GEMINI_MODEL)
print("Ollama model:", OLLAMA_MODEL)


Model clients configured.
Gemini model: gemini-2.5-flash
Ollama model: llama3.1:8b


In [4]:
# Using the same question helps make the comparison fair.
# We will later change the system message while keeping the user question unchanged.

# The same user question will be sent to both models.
USER_QUESTION = """
I have been asked to analyze a drop in customer retention
for a subscription product. What should I investigate first,
and what data would I need?
"""

# A neutral system message for the baseline comparison.
BASELINE_SYSTEM = """
You are a helpful assistant supporting a data science team.
Give practical advice and explain what data is needed.
"""

print("Shared question and baseline system message are ready.")


Shared question and baseline system message are ready.


## Part 2: Basic Query Function

**TODO:**
Create a reusable function or workflow that sends a user message to each model.

Your code should make it easy to change the user message and rerun the comparison.

At minimum, your code should:

- accept a user message
- send the message to both model access methods
- capture the responses
- display the responses clearly

**Q3:** What differences did you notice in the setup process for the two access methods?

**A3:**

The two methods differ in almost every setup step. Gemini required creating an API key in Google AI Studio, storing it as a secret, and installing one SDK; after that a single call worked immediately. Nothing had to be downloaded and the compute happened remotely.

Ollama required installing a desktop application, downloading a ~5 GB model file, and keeping a background server process running. In exchange there was no account, no key, no quota, and no per-request cost. The first call to the local model is also noticeably slower than later ones because the weights have to be loaded into memory, so I added a warm-up call before measuring latency.

An earlier attempt to use the Hugging Face Inference API as Model 2 failed with a 403 permission error (gated model, no inference credits), which is what led me to the local option. That experience highlighted a general point: hosted "open" models still depend on a provider's availability and billing, whereas a local runtime depends only on the hardware in front of you.


In [9]:
# Code for PART 2
# This function avoids repeating the API code throughout the notebook. It also records latency and returns errors for inspection.
# Gemini latency includes network and service time; Ollama latency is pure on-device inference time on this Mac.

TEMPERATURE = 0.2
MAX_TOKENS = 700


def query_model(provider, question, system_message=None):
    """
    Send a question to Gemini (cloud API) or Ollama (local runtime).

    Returns a dictionary containing:
      provider, model, answer, latency_seconds, error
    """
    start_time = time.perf_counter()

    try:
        if provider == "gemini":
            response = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=question,
                                config={
                    "system_instruction": system_message or "You are a helpful assistant.",
                    "temperature": TEMPERATURE,
                    "max_output_tokens": MAX_TOKENS,
                    "thinking_config": {"thinking_budget": 0},
                },
            )
            answer = response.text or "No text response returned."
            model_name = GEMINI_MODEL

        elif provider == "ollama":
            messages = []
            if system_message:
                messages.append({"role": "system", "content": system_message})
            messages.append({"role": "user", "content": question})

            response = ollama.chat(
                model=OLLAMA_MODEL,
                messages=messages,
                options={"temperature": TEMPERATURE, "num_predict": MAX_TOKENS},
            )
            answer = response.message.content or "No text response returned."
            model_name = OLLAMA_MODEL

        else:
            raise ValueError("Provider must be 'gemini' or 'ollama'.")

        elapsed = time.perf_counter() - start_time
        return {
            "provider": provider,
            "model": model_name,
            "answer": answer,
            "latency_seconds": round(elapsed, 2),
            "error": None,
        }

    except Exception as e:
        # Record errors rather than stopping the whole notebook.
        elapsed = time.perf_counter() - start_time
        return {
            "provider": provider,
            "model": GEMINI_MODEL if provider == "gemini" else OLLAMA_MODEL,
            "answer": "",
            "latency_seconds": round(elapsed, 2),
            "error": str(e),
        }


def compare(question, system_message=None, label=""):
    """Send one question to both models and display the results side by side."""
    results = [
        query_model("gemini", question, system_message),
        query_model("ollama", question, system_message),
    ]
    for r in results:
        print("=" * 80)
        print(f"{label} {r['provider'].upper()} ({r['model']})")
        print(f"Latency: {r['latency_seconds']} s   Error: {r['error']}")
        print("-" * 80)
        print(r["answer"])
    print("=" * 80)
    return results


## Part 3: Shared Task Comparison

Use the following scenario as the shared task for your model comparison:

    You are building an internal assistant for a data science team. The assistant helps analysts decide how to approach a new project. A user asks:

    “I have been asked to analyze a drop in customer retention for a subscription product. What should I investigate first, and what data would I need?”

You may modify the system message, but the user request should remain the same for all comparisons.

**TODO:**
- Display the model response to the scenario clearly.
- Then, complete the following table with scores of 1-5 for each criterion
  

| Criterion	| Model 1 | Model2|
| --- | --- | --- |
| Relevance to the user question | 5| 5|
| Specificity of recommendations | 5| 4|
| Usefulness for a data scientist | 5| 4|
| Clarity and organization | 5| 4|
| Potential risks or limitations| 4| 3|

**Q4:** Which response was stronger overall? What specific evidence supports your judgment?

**A4:**

Gemini’s response was stronger overall. Both answers were on topic and covered the same broad areas (quantify the drop, segment customers, gather usage/billing data), but Gemini framed the problem the way a data scientist actually would: it led with cohort retention by signup month and an explicit new vs existing segmentation, which is the correct first analytical move, and it organized the answer into a clear “what to investigate / why / data needed” hierarchy. 

Llama’s answer was solid and well-structured but read more like a generic data checklist (customer data, behavioral data, segmentation data) with a CRR formula, rather than a prioritized analytical plan. Gemini also produced more content in far less time — 448 words in 5.83 s versus Llama’s 344 words in 21.27 s. The one area where neither excelled was surfacing its own limitations or caveats; Gemini did slightly better by distinguishing a sudden drop (specific event) from a gradual decline (systemic issue).

In [13]:
# Code for PART 3

# Run the same task through both models with the same neutral system message.
baseline_results = compare(USER_QUESTION, BASELINE_SYSTEM, label="BASELINE")
gemini_baseline, ollama_baseline = baseline_results

# Summary table of the baseline run (used in Part 5).
baseline_df = pd.DataFrame([
    {"model": r["model"], "latency_s": r["latency_seconds"],
     "words": len(r["answer"].split()), "error": r["error"]}
    for r in baseline_results
])
baseline_df


BASELINE GEMINI (gemini-2.5-flash)
Latency: 4.03 s   Error: None
--------------------------------------------------------------------------------
This is a classic problem in subscription businesses, and a critical one to solve! A drop in customer retention can significantly impact long-term revenue and growth.

Here's a practical approach to what you should investigate first, along with the data you'll need.

## What to Investigate First: The "When" and "Where"

Before diving into *why* retention dropped, you need to precisely define *when* and *where* the drop occurred. This will help you narrow down potential causes.

**1. Define the Drop:**
   * **When did the drop start?** Pinpoint the exact date or week when the retention rate began to decline. This is your "event horizon."
   * **What was the magnitude of the drop?** How much did it decrease? (e.g., from 85% to 80% monthly retention).
   * **Is it a sustained drop or a temporary blip?** A sustained drop is more concerning.

**2.

,model,latency_s,words,error
0,gemini-2.5-flash,4.03,380,None
1,llama3.1:8b,24.99,428,None


## Part 4: System Message Experiment

**TODO:**

Adjust the system message to define a specific role. For example:

    You are a senior data scientist advising a junior analyst. Give practical, prioritized advice. Ask clarifying questions when needed. Avoid making unsupported assumptions.

Run the same user request again using your own system message (*NOT THE PROVIDED EXAMPLE*).

**Q5:** How did the system message change the model outputs? Did one model respond more strongly to the system message than the other?

**A5:**

The role based system message changed both models, but Gemini responded more strongly. My prompt demanded a fixed three-part structure (clarifying questions, a prioritized checklist with one data source per item, one sentence on what to avoid) and a 300-word limit. Gemini followed all of it: it dropped from 448 to 217 words, asked exactly two clarifying questions, and attached a specific named data source to each checklist item (customer_churn_log, billing_events, support_ticket_log). Llama also adopted the structure and the “what not to do” line, but shrank far less (344 to 273 words) and stayed closer to its baseline verbosity. In other words, the stronger model was also the more steerable one — it honored both the format and the length constraint, while the local 8B model honored the format but largely ignored the word cap.

In [11]:
# Code for PART 4

# A custom role-based system message (not the provided example).
# The user question stays exactly the same as in Part 3.
ROLE_SYSTEM = """
You are the analytics lead for a subscription-software company, mentoring an
analyst who has three days to deliver a first read on a retention drop.
Respond in this structure:
1. Two or three clarifying questions you would ask before starting.
2. A prioritized checklist (most likely causes first) with the ONE table or
metric needed to test each item.
3. One sentence on what you would NOT spend time on in the first three days.
Be concrete, use the company's vocabulary (cohorts, churn, MRR, plan tiers),
and keep the whole answer under 300 words.
"""

role_results = compare(USER_QUESTION, ROLE_SYSTEM, label="ROLE PROMPT")
gemini_role, ollama_role = role_results

# Compare how each model's output changed between the two system messages.
pd.DataFrame([
    {"model": b["model"],
    "baseline_latency_s": b["latency_seconds"], "role_latency_s": r["latency_seconds"],
    "baseline_words": len(b["answer"].split()), "role_words": len(r["answer"].split())}
    for b, r in zip(baseline_results, role_results)
])


ROLE PROMPT GEMINI (gemini-2.5-flash)
Latency: 2.64 s   Error: None
--------------------------------------------------------------------------------
Great question! Let's get you set up for success on this.

1.  **Clarifying Questions:**
    *   When did this retention drop become noticeable, and for which specific cohorts (e.g., Q1 2024 sign-ups)?
    *   Has there been any recent change in product features, pricing, or marketing campaigns that might coincide with this drop?

2.  **Prioritized Checklist & Data:**

    *   **Churn Spike for Specific Cohorts/Plan Tiers:** Is the overall retention drop driven by a significant increase in churn for a particular group of customers?
        *   **Data:** `customer_churn_log` (filtered by `churn_date`, `cohort_signup_date`, `plan_tier`)
    *   **Onboarding Experience Degradation:** Are newer customers failing to activate or see value early on?
        *   **Data:** `user_activity_log` (time to first key action, feature adoption rate for new

,model,baseline_latency_s,role_latency_s,baseline_words,role_words
0,gemini-2.5-flash,5.83,2.64,448,217
1,llama3.1:8b,21.27,14.41,344,273


## Part 5: Cost, Latency, and Usability Reflection

**Q6:** For each access method, record basic practical information.

At minimum, include:
- approximate latency, measured informally or programmatically
- whether the service required an API key
- whether there were usage limits, credits, or costs
- ease of setup
- any privacy or data-sharing considerations

**A6:**

| | Gemini 2.5 Flash (cloud API) | Llama 3.1 8B (local, Ollama) |
| --- | --- | --- |
| Approx. latency | ~3–6 s (baseline 5.83 s, role 2.64 s) | ~14–21 s (baseline 21.27 s, role 14.41 s) |
| API key required | Yes (Google AI Studio) | No |
| Usage limits / cost | Free-tier daily request/rate limits; billable above the free tier | None — only local disk (~5 GB) and RAM |
| Ease of setup | Easy: one key, one SDK, worked immediately | More involved: install Ollama, run a local server, pull a 5 GB model |
| Privacy / data-sharing | Every prompt leaves the machine and is sent to Google's servers, subject to their data-use terms | Fully offline; no prompt ever leaves the Mac |

Latency is the sharpest trade-off: the cloud model was 3–4× faster because all the compute happens on Google's hardware, whereas the local model runs entirely on my MacBook Air's CPU/GPU. Setup inverts that: Gemini was trivial to connect, while Ollama required a background server and a large model download. The decisive differences are cost and privacy — the local model has no per-request cost, no quota, and sends nothing over the network.



**Q7:** If you were building a prototype for a real organization, which access method would you choose and why? Describe a hypothetical organization that you are building for in your answer.

**A7:**

For a hypothetical organization I’d pick based on the data sensitivity constraint. Imagine an automotive telematics supplier building an internal assistant that engineers query with logs, diagnostic traces, and unreleased firmware details. For that org I’d choose the local model (Ollama), despite its higher latency, because the proprietary and potentially regulated data must never leave company-controlled hardware, and an on device model gives offline operation, zero per query cost, and full reproducibility (fixed weights, no silent provider side model updates). If instead the org were a small marketing startup drafting public facing copy with no sensitive data and tight deadlines, I’d choose Gemini for its speed, stronger output quality, and near-zero setup. The deciding factor is not raw quality — where Gemini wins — but whether the data can legally and safely be sent to a third party.

## Part 6: AI Assistance Reflection

Because LLM engineering often involves using AI tools to write, debug, and revise code, you are expected to use an AI assistant in this assignment. However, you are also expected to evaluate and take responsibility for the work it produces.

Answer the following questions.

**Q8:** What AI assistant did you use during this assignment?

**A8:**
Claude (Anthropic).

**Q9:** What parts of the assignment did the assistant help with?

**A9:**

Helped  diagnose errors, and structure the comparison code.

**Q10:** Describe one suggestion, explanation, or code snippet from the assistant that was useful.

**A10:**

The compare() helper that sends one question to both models and prints them side by side with latency, plus the latency/word-count DataFrame — made the comparison repeatable.

**Q11:** Describe one thing the assistant got wrong, left unclear, or failed to consider.

**A11:**

The assistant’s initial setup didn’t account for Gemini 2.5 Flash’s thinking tokens counting against max_output_tokens, which silently truncated Model 1 to ~20 words. Had I not caught it, I’d have wrongly concluded the local model was far more thorough. The fix was adding thinking_config={"thinking_budget": 0}.

**Q12:** What did you change, verify, or decide yourself?

**A12:**

I installed Ollama and the model myself, chose llama3.1:8b, ran everything locally, verified the Gemini truncation by inspecting word counts, re-ran after the fix, and assigned the comparison scores based on my own reading of the outputs.